# Lab 2.2 &mdash; ReAct and the Parser Contract

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Parse Thought / Action / Observation out of free-form model text
- Break your own parser on eight real drift cases
- Choose between strict and lenient, and defend the choice
- Drive the Module 1 loop with the parser you wrote

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Builds on Lab 1.1.** You wrote the loop; here you write the thing that turns the
> model's prose into an actual tool call, every single time, or fails safely.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
LLM_MODEL    = os.environ.get("LAB_LLM_MODEL")    or os.environ.get("OPENAI_MODEL")
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 1.2 of Module 1
# These are the tools you wrote in Lab 1.2 of Module 1. Nothing to fill in -- they are here so this
# notebook runs on its own. Note the docstrings: they name the case AND the boundary.

def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}
print("carried forward:", ", ".join(TOOLS))

## Concept

ReAct interleaves reasoning with tool use: **Thought** &rarr; **Action** &rarr; **Observation**, and
round again. The loop is Module 1's. What is new is that the model's *text* must become a *call*.

That parser is a **protocol boundary between a system that guarantees its output format and one
that does not.** It will meet malformed input in production. The only question is what it does then.

## Section 1 &mdash; The happy path

Start with well-formed output. `parse_step` returns a dict describing what the model wants.

In [ ]:
import re

def parse_step(text: str) -> dict:
    """Parse one ReAct step.

    Returns one of:
      {"kind": "action", "tool": str, "arg": str, "thought": str}
      {"kind": "final",  "answer": str, "thought": str}
      {"kind": "unparseable", "raw": str}
    """
    thought = ""
    m = re.search(r"Thought:\**\s*(.+)", text)
    if m:
        thought = m.group(1).strip()

    m = re.search(r"Final Answer:\**\s*(.+)", text, re.S)
    if m:
        return {"kind": "final", "answer": m.group(1).strip(), "thought": thought}

    m = re.search(r'Action:\**\s*(\w+)\s*\(\s*"([^"]*)"\s*\)', text)
    if m:
        return {"kind": "action", "tool": m.group(1), "arg": m.group(2), "thought": thought}
    return {"kind": "unparseable", "raw": text}

In [ ]:
# --- Self-check: Section 1
_good = 'Thought: I need the record.\nAction: lookup_payment("PMT-1002")'
_final = 'Thought: I have enough.\nFinal Answer: retry once after 24h'

check("a well-formed action parses", lambda: parse_step(_good)["kind"] == "action")
check("the tool name is extracted", lambda: parse_step(_good)["tool"] == "lookup_payment")
check("the argument is extracted", lambda: parse_step(_good)["arg"] == "PMT-1002")
check("the thought is kept", lambda: "record" in parse_step(_good)["thought"])
check("a final answer parses as final", lambda: parse_step(_final)["kind"] == "final")
check("noise is reported, not raised", lambda: parse_step("hello")["kind"] == "unparseable")

## Section 2 &mdash; Eight ways it drifts

Every one of these is something a real model emits. Run the cell and see how many your parser
survives &mdash; the point is not to score well, it is to see the shape of the problem.

In [ ]:
DRIFT = [
    ("bold markers",      '**Thought:** need it\n**Action:** lookup_payment("PMT-1002")'),
    ("unquoted argument", 'Action: lookup_payment(PMT-1002)'),
    ("single quotes",     "Action: lookup_payment('PMT-1002')"),
    ("prose argument",    'Action: lookup_payment for payment PMT-1002'),
    ("two actions",       'Action: lookup_payment("PMT-1002")\nAction: policy_for("X")'),
    ("thought only",      'Thought: I should probably check the ledger first.'),
    ("fenced in code",    '```\nAction: lookup_payment("PMT-1002")\n```'),
    ("trailing chatter",  'Action: lookup_payment("PMT-1002")\nLet me know if you need more!'),
]

print(f"{'drift case':22}{'parsed as':16}{'tool':18}arg")
for name, text in DRIFT:
    try:
        r = parse_step(text)
        print(f"{name:22}{r['kind']:16}{r.get('tool',''):18}{r.get('arg','')}")
    except NameError:
        print("(fill in parse_step above, then re-run)"); break

In [ ]:
# --- Self-check: Section 2   (what a STRICT parser must and must not do)
def kind_of(text):
    return parse_step(text)["kind"]

check("bold markers still parse -- the regex is not anchored to line start",
      lambda: kind_of(DRIFT[0][1]) == "action")
check("an unquoted argument is refused rather than guessed",
      lambda: kind_of(DRIFT[1][1]) == "unparseable",
      "a strict parser must not invent the argument it did not see")
check("a thought with no action is refused",
      lambda: kind_of(DRIFT[5][1]) == "unparseable")
check("a fenced action still parses", lambda: kind_of(DRIFT[6][1]) == "action")
check("trailing chatter does not break the action",
      lambda: parse_step(DRIFT[7][1])["arg"] == "PMT-1002")
check("two actions in one turn takes only the first",
      lambda: parse_step(DRIFT[4][1])["arg"] == "PMT-1002",
      "executing both is how one turn becomes two side effects")

## Section 3 &mdash; Strict or lenient?

Now the judgement. A **lenient** parser accepts the unquoted argument and keeps the run going. A
**strict** one refuses and asks the model to restate. Both are defensible; they fail differently.

In [ ]:
def parse_lenient(text: str) -> dict:
    """Strict first; if that fails, accept a bare unquoted argument."""
    r = parse_step(text)
    if r["kind"] != "unparseable":
        return r
    m = re.search(r"Action:\**\s*(\w+)\s*\(\s*([^)\"']+?)\s*\)", text)
    if m:
        return {"kind": "action", "tool": m.group(1), "arg": m.group(2).strip(), "thought": ""}
    return r

def parser_for(tool_kind: str) -> str:
    """Which parser should drive this kind of tool: "strict" or "lenient"?

    strict  -- refuse anything ambiguous and ask the model to restate
    lenient -- keep the run going, accepting a best guess at the argument
    """
    if tool_kind == "write":
        return "strict"              # a wrong guess here is an irreversible action
    return "lenient"                 # a wrong read costs one step, and the agent recovers

In [ ]:
# --- Self-check: Section 3
check("the lenient parser recovers the unquoted argument",
      lambda: parse_lenient(DRIFT[1][1])["arg"] == "PMT-1002")
check("the lenient parser still refuses a thought with no action",
      lambda: parse_lenient(DRIFT[5][1])["kind"] == "unparseable",
      "lenient means tolerant of format, not of missing intent")
check("writes are driven by the strict parser",
      lambda: parser_for("write") == "strict",
      "a misparsed write is irreversible; a refusal costs one retry")
check("reads can afford the lenient parser",
      lambda: parser_for("read") == "lenient",
      "a wrong read wastes a step and the agent recovers from the observation")

## Section 4 &mdash; Drive the loop

Wire the parser into a ReAct loop with Module 1's budget and stop conditions. Deterministic here:
a scripted model, so the loop is exercised without a live call.

In [ ]:
def react_loop(steps, tools, max_steps=6):
    """steps: a list of model replies, replayed in order. Returns the run state."""
    state = {"steps": 0, "answer": None, "trace": [], "stopped": None}
    for reply in steps:
        if state["steps"] >= max_steps:
            state["stopped"] = "budget"
            return state
        r = parse_step(reply)
        if r["kind"] == "final":
            state["answer"] = r["answer"]
            state["stopped"] = "goal"
            return state
        if r["kind"] == "unparseable":
            state["trace"].append(("unparseable", reply[:30], "asked to restate"))
            state["steps"] += 1
            continue
        fn = tools.get(r["tool"])
        obs = fn(r["arg"]) if fn else f"no such tool {r['tool']!r}"
        state["trace"].append((r["tool"], r["arg"], obs))
        state["steps"] = state["steps"] + 1
    state["stopped"] = state["stopped"] or "ran out of replies"
    return state

In [ ]:
# --- Self-check: Section 4
_script = [
    'Thought: get the record.\nAction: lookup_payment("PMT-1003")',
    'Thought: now the policy.\nAction: policy_for("LIMIT_BREACH")',
    'Thought: done.\nFinal Answer: Treasury must approve before release.',
]
def _run():
    return react_loop(_script, TOOLS)

check("the run reaches its goal", lambda: _run()["stopped"] == "goal")
check("both tool calls are traced", lambda: len(_run()["trace"]) == 2)
check("the observation carries the real ledger data",
      lambda: "LIMIT_BREACH" in _run()["trace"][0][2])
check("the final answer is captured", lambda: "Treasury" in _run()["answer"])
check("an unparseable step costs budget but does not stop the run",
      lambda: react_loop(["garbage"] + _script, TOOLS)["stopped"] == "goal",
      "state['steps'] must advance on the unparseable branch too")
check("a script that never finishes stops on the budget",
      lambda: react_loop(['Action: lookup_payment("PMT-1002")'] * 10, TOOLS)["stopped"] == "budget")

## Run it for real

A live model, your parser, real tools. Watch the format the model actually chooses &mdash; you did not
specify it, so it picked one.

In [ ]:
REACT_SYSTEM = """You investigate payment exceptions. Work in this exact format:

Thought: <one line of reasoning>
Action: <tool>("<argument>")

Available tools:
  lookup_payment("PMT-1002")  -- returns the ledger record
  policy_for("LIMIT_BREACH")  -- returns the policy for a reason code

After an Observation, continue with another Thought/Action, or finish with:
Final Answer: <what should happen>

Emit exactly one Thought and one Action per turn."""

if llm_ready():
    try:
        transcript, state = [], {"steps": 0}
        convo = "Investigate PMT-1005."
        for turn in range(5):
            reply = ask(convo, system=REACT_SYSTEM)
            r = parse_step(reply)
            print(f"--- turn {turn + 1} [{r['kind']}]")
            print("   " + reply.strip().replace("\n", "\n   ")[:240])
            if r["kind"] == "final":
                break
            if r["kind"] == "unparseable":
                convo += "\nObservation: could not parse that. Reply with exactly one Thought and one Action."
                continue
            fn = TOOLS.get(r["tool"])
            obs = fn(r["arg"]) if fn else f"no such tool {r['tool']!r}"
            print(f"   Observation: {obs[:160]}")
            convo += f"\n{reply}\nObservation: {obs}"
    except NameError:
        print("(fill in the blanks above, then re-run this cell)")

### Read the trace

Count the `[unparseable]` turns. Zero means this model happens to follow your format today &mdash; not
that your parser is safe. The drift table in Section 2 is what it looks like when that changes,
which it does with every model swap and most prompt edits.

Notice too that the recovery path matters: an unparseable turn feeds a corrective observation back
rather than crashing. That is the same "tools report failure, they do not raise" discipline from
Module 1, applied to the model's own output.

In [ ]:
score()

## Your turn

1. Add a ninth drift case from your own experience and decide, with a reason, whether strict or
   lenient should handle it.
2. `react_loop` gives an unparseable turn the same budget cost as a real step. Should it? Argue
   both sides &mdash; then consider what a separate, smaller "reformat" budget would buy you.